# G5 — Stage 8A XAI: Purity, AP, Faithfulness, Stability

Stage 8A: ProtoSegNet L4 only, with skip (single_scale=True)  
Checkpoint: `proto_seg_ct_abl_a.pth`  
Projection file `projected_prototypes_ct_l4only.pt` norms=13.42 vs ckpt norms=10.42 → stale, fresh projection needed.

Known from v8 ablation: Dice=0.810, AP=0.030, Stability=2.46  
Missing: Purity, Faithfulness (pixel + patch)

In [1]:
import sys, os
_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(_root)
sys.path.insert(0, _root)
os.environ.setdefault('PYTORCH_MPS_HIGH_WATERMARK_RATIO', '0.0')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

import torch
import torch.nn as nn
import torch.utils.data as tud
import numpy as np
import pandas as pd
from pathlib import Path

from src.data.mmwhs_dataset import MMWHSSliceDataset, NUM_CLASSES
from src.models.proto_seg_net import ProtoSegNet
from src.models.prototype_layer import PrototypeProjection
from src.metrics.proto_quality import (
    compute_purity, compute_per_level_ap, compute_compactness,
    compute_level_dominance, compute_effective_quality,
)
from src.metrics.faithfulness import faithfulness_patient
from src.metrics.stability import stability_patient
from src.metrics.patch_faithfulness import patch_faithfulness_patient

DEVICE   = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
DATA_DIR = 'data/pack/processed_data'
CKPT_DIR = Path('checkpoints')
OUT_DIR  = Path('results/v10')
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_SLICES = 50

class Adapter(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
    def forward(self, x, **kw):
        logits, hm = self.base(x)
        n = len(self.base.proto_levels)
        w = torch.full((x.size(0), n), 1.0/n, device=x.device)
        return logits, hm, w

print(f'Device: {DEVICE}')

Device: mps


## 1. Load checkpoint (no projection file)

In [2]:
CKPT = CKPT_DIR / 'proto_seg_ct_abl_a.pth'
ckpt = torch.load(CKPT, map_location='cpu', weights_only=False)
print(f'Checkpoint: epoch={ckpt["epoch"]}  best_val={ckpt["best_val_dice"]:.4f}')
print(f'single_scale={ckpt.get("single_scale")}  no_soft_mask={ckpt.get("no_soft_mask")}')

model = ProtoSegNet(
    n_classes=NUM_CLASSES,
    single_scale=True,          # L4 only
    no_soft_mask=ckpt.get('no_soft_mask', False),
).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

for l in model.proto_levels:
    p = model.proto_layers[str(l)].prototypes.data
    norms = p.reshape(-1, p.shape[-1]).norm(dim=-1)
    print(f'  L{l} ckpt norms: mean={norms.mean():.2f}  min={norms.min():.2f}  max={norms.max():.2f}')
print('Model ready.')

Checkpoint: epoch=45  best_val=0.8104
single_scale=True  no_soft_mask=False
  L4 ckpt norms: mean=10.42  min=4.92  max=12.63
Model ready.


## 2. Fresh prototype projection

In [3]:
FRESH_PROJ = CKPT_DIR / 'projected_prototypes_ct_abl_a_fresh.pt'

train_ds = MMWHSSliceDataset(DATA_DIR, 'ct', 'train', augment=False, preload=True)
proj_loader = tud.DataLoader(train_ds, batch_size=32, shuffle=False)
wrapped = [(b['image'], b['label']) for b in proj_loader]
print(f'Training batches: {len(wrapped)}')

projector = PrototypeProjection(
    encoder=model.encoder,
    proto_layers=model.proto_layers_dict(),
    device='cpu',
)
print('Running projection (CPU)…')
projector.project(wrapped, save_path=str(FRESH_PROJ))
print(f'Saved → {FRESH_PROJ.name}')

proj = torch.load(FRESH_PROJ, map_location='cpu', weights_only=True)
for level, proto_data in proj['proto_state'].items():
    model.proto_layers[str(level)].prototypes.data.copy_(proto_data)

for l in model.proto_levels:
    p = model.proto_layers[str(l)].prototypes.data
    norms = p.reshape(-1, p.shape[-1]).norm(dim=-1)
    print(f'  L{l} post-projection norms: mean={norms.mean():.2f}')

model.to(DEVICE)
model.eval()
adapted = Adapter(model).to(DEVICE)
adapted.eval()

Training batches: 106
Running projection (CPU)…


Projected prototypes saved → checkpoints/projected_prototypes_ct_abl_a_fresh.pt
Saved → projected_prototypes_ct_abl_a_fresh.pt
  L4 post-projection norms: mean=11.84


Adapter(
  (base): ProtoSegNet(
    (encoder): HierarchicalEncoder2D(
      (level1): EncoderBlock(
        (down): Sequential(
          (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (res): ResBlock(
          (conv): Sequential(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
            (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
          (shortcut): Identity()
          (relu): ReLU(inplace=True)
        )
      )
      (level2): EncoderBlock(
        (down): Sequential(
        

## 3. Data loaders

In [4]:
train_loader = tud.DataLoader(
    MMWHSSliceDataset(DATA_DIR, 'ct', 'train', augment=False, preload=True),
    batch_size=16, shuffle=False
)
test_loader = tud.DataLoader(
    MMWHSSliceDataset(DATA_DIR, 'ct', 'test', augment=False, preload=True),
    batch_size=16, shuffle=False
)
test_ds = MMWHSSliceDataset(DATA_DIR, 'ct', 'test', augment=False, preload=True)
imgs_test = torch.stack([test_ds[i]['image'] for i in range(len(test_ds))])
print(f'Train: {len(train_loader.dataset)} slices  Test: {len(test_loader.dataset)} slices')

Train: 3389 slices  Test: 484 slices


## 4. Purity + AP

In [5]:
print('Computing purity…')
purity_df = compute_purity(model, train_loader)
print('Computing AP…')
ap_df = compute_per_level_ap(model, test_loader)
print('Computing compactness…')
compact_df = compute_compactness(model, test_loader)
print('Computing level dominance…')
dom_df = compute_level_dominance(model, test_loader)
print('Computing effective quality…')
eq_df = compute_effective_quality(purity_df, ap_df, compact_df, dom_df)

purity_df.to_csv(OUT_DIR / 'xai_purity_8a.csv', index=False)
ap_df.to_csv(OUT_DIR / 'xai_ap_8a.csv', index=False)
eq_df.to_csv(OUT_DIR / 'xai_effective_quality_8a.csv', index=False)

print('\n─── Per-level Purity ───')
print(purity_df.groupby('level')[['purity']].mean().round(3).to_string())
print('\n─── Per-level AP ───')
print(ap_df.groupby('level')[['ap']].mean().round(3).to_string())
print('\n─── Effective quality ───')
print(eq_df.round(3).to_string(index=False))

Computing purity…


Computing AP…


Computing compactness…


Computing level dominance…


Computing effective quality…

─── Per-level Purity ───
       purity
level        
4       0.474

─── Per-level AP ───
          ap
level       
4      0.057

─── Effective quality ───
 weight_l4  purity_l4  ap_l4  compact_l4  effective_purity  effective_ap  effective_compactness
       1.0      0.474  0.057       0.777             0.474         0.057                  0.777


## 5. Faithfulness + Stability + Patch Faithfulness

In [6]:
print('Computing Faithfulness…')
faith = faithfulness_patient(adapted, imgs_test, DEVICE, max_slices=MAX_SLICES)
print(f'  Faithfulness : {faith["faithfulness"]:.4f}  (std {faith["faithfulness_std"]:.4f})')

print('Computing Stability…')
stab = stability_patient(adapted, imgs_test, DEVICE, max_slices=MAX_SLICES)
print(f'  Stability    : {stab["stability"]:.4f}  (std {stab["stability_std"]:.4f})')

print('Computing Patch Faithfulness (bs=16)…')
pfaith = patch_faithfulness_patient(adapted, imgs_test, DEVICE, block_size=16, max_slices=MAX_SLICES)
print(f'  Patch Faith  : {pfaith["patch_faithfulness"]:.4f}  (std {pfaith["patch_faithfulness_std"]:.4f})')

pd.DataFrame([faith]).to_csv(OUT_DIR / 'xai_faithfulness_8a.csv', index=False)
pd.DataFrame([stab]).to_csv(OUT_DIR / 'xai_stability_8a.csv', index=False)
pd.DataFrame([pfaith]).to_csv(OUT_DIR / 'xai_patch_faithfulness_8a.csv', index=False)

Computing Faithfulness…


  Faithfulness : 0.0928  (std 0.0392)
Computing Stability…


  Stability    : 3.7855  (std 0.6576)
Computing Patch Faithfulness (bs=16)…


  Patch Faith  : 0.1611  (std 0.0891)


## 6. Summary

In [7]:
eff_pur  = float(eq_df['effective_purity'].iloc[0])
eff_ap   = float(eq_df['effective_ap'].iloc[0])
eff_comp = float(eq_df['effective_compactness'].iloc[0])
faith_v  = faith['faithfulness']
stab_v   = stab['stability']
pfaith_v = pfaith['patch_faithfulness']
val_dice = ckpt['best_val_dice']

rows = [
    ('Val Dice',           val_dice, None, None, '—'),
    ('Effective Purity',   eff_pur,  None, None, '—'),
    ('Effective AP',       eff_ap,   0.15, 0.25, '✅' if eff_ap  >= 0.15 else '❌'),
    ('Effective Compact.', eff_comp, None, None, '—'),
    ('Faithfulness (px)',  faith_v,  0.15, 0.30, '✅' if faith_v >= 0.15 else '❌'),
    ('Stability',          stab_v,   None, 2.00, '✅' if stab_v  <= 2.00 else '❌'),
    ('Patch Faith (bs16)', pfaith_v, None, None, '—'),
]
summary = pd.DataFrame(rows, columns=['Metric', 'Value', 'Min gate', 'Target', 'Pass'])
summary.to_csv(OUT_DIR / 'xai_summary_8a.csv', index=False)

print('Stage 8A (skip, L4 only) — XAI Summary')
print('=' * 60)
print(summary.to_string(index=False))
print()

# Complete 2×2 table
print('=' * 75)
print('  Complete 2×2 Table: Skip × Level')
print('=' * 75)
print(f'  {"Metric":<22}  {"Stage 29":>12}  {"Stage 8A":>12}  {"9b":>10}  {"9a":>10}')
print(f'  {"":.<22}  {"skip,L3+L4":>12}  {"skip,L4":>12}  {"noskip,L3+L4":>10}  {"noskip,L4":>10}')
print('  ' + '─' * 71)
data = [
    ('Val Dice',       0.821,    val_dice,  0.559,  0.606),
    ('Eff. Purity',    0.527,    eff_pur,   0.686,  0.679),
    ('Eff. AP',        0.051,    eff_ap,    0.301,  0.312),
    ('Faithfulness',   0.069,    faith_v,   0.035,  0.012),
    ('Stability',      3.382,    stab_v,    11.94,  10.92),
    ('Patch Faith',    0.212,    pfaith_v,  0.200,  0.259),
]
for name, v29, v8a, v9b, v9a in data:
    print(f'  {name:<22}  {v29:>12.3f}  {v8a:>12.3f}  {v9b:>10.3f}  {v9a:>10.3f}')
print('=' * 75)

Stage 8A (skip, L4 only) — XAI Summary
            Metric    Value  Min gate  Target Pass
          Val Dice 0.810354       NaN     NaN    —
  Effective Purity 0.474286       NaN     NaN    —
      Effective AP 0.057137      0.15    0.25    ❌
Effective Compact. 0.777083       NaN     NaN    —
 Faithfulness (px) 0.092814      0.15    0.30    ❌
         Stability 3.785521       NaN    2.00    ❌
Patch Faith (bs16) 0.161133       NaN     NaN    —

  Complete 2×2 Table: Skip × Level
  Metric                      Stage 29      Stage 8A          9b          9a
  ......................    skip,L3+L4       skip,L4  noskip,L3+L4   noskip,L4
  ───────────────────────────────────────────────────────────────────────
  Val Dice                       0.821         0.810       0.559       0.606
  Eff. Purity                    0.527         0.474       0.686       0.679
  Eff. AP                        0.051         0.057       0.301       0.312
  Faithfulness                   0.069         0.093    